# Phase 2: RAG Application (The Interface)

This notebook is the **"User Interface"** for your data.
It does NOT ingest files. It only reads the Database that Phase 1 created on your E: drive.

### Goals:
1. Connect to the Chroma Database on E: (Read-Only).
2. Connect to the Grok/Gemini LLM.
3. Perform RAG (Retrieval Augmented Generation) to answer questions about WMS selection.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

# Load API Keys
load_dotenv(override=True)

# Disable parallelism warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# --- CONFIGURATION ---

# 1. Database Path (Must match Phase 1)
DB_PATH = "/mnt/e/chroma_db_wms"

# 2. Embedding Model (Must match Phase 1)
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# 3. LLM Configuration (Grok or Gemini)
LLM_MODEL = "grok-2-latest" # or "grok-4-1-fast-reasoning"
LLM_BASE_URL = os.getenv("GROK_BASE_URL", "https://api.x.ai/v1")
LLM_API_KEY = os.getenv("GROK_API_KEY")

print(f"Connecting to Database at: {DB_PATH}")

In [ ]:
# --- 1. CONNECT TO DATABASE ---

# We use the same HuggingFace model to "search" that we used to "save"
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

vectorstore = Chroma(
    persist_directory=DB_PATH,
    embedding_function=embeddings
)

print(f"Connected! Loaded {vectorstore._collection.count():,} documents from the vault.")

# Convert to a Retriever tool (It will fetch the top 5 most relevant chunks)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [ ]:
# --- 2. SETUP LLM ---

if not LLM_API_KEY:
    raise ValueError("GROK_API_KEY not found in .env file")

llm = ChatOpenAI(
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY,
    model=LLM_MODEL,
    temperature=0.0 # Low temperature for factual accuracy
)

print("LLM Ready.")

In [ ]:
# --- 3. DEFINE THE RAG CHAIN ---

# The Prompt Template
template = """
You are an expert Warehouse Management System (WMS) consultant.
Answer the user's question based ONLY on the context provided below.
If the answer is not in the context, say "I don't see that information in the documents."
Cite the source file names if possible.

Context:
{context}

Question:
{question}
"""

prompt = ChatPromptTemplate.from_template(template)

# The Chain
def format_docs(docs):
    return "\n\n".join([f"[Source: {d.metadata.get('source', 'Unknown')}]\n{d.page_content}" for d in docs])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# --- 4. CHAT WITH YOUR DATA ---

question = "What are the main functional requirements for the WMS?"

print(f"Question: {question}\n")
print("Analyzing 105GB of documents... (this retrieves the relevant parts instantly)")

response = rag_chain.invoke(question)

print("\n--- ANSWER ---")
print(response)

In [ ]:
# Interactive Loop (Run this cell to keep chatting)
while True:
    q = input("\nAsk a question (or 'q' to quit): ")
    if q.lower() in ['q', 'quit', 'exit']:
        break
    
    print("Thinking...")
    resp = rag_chain.invoke(q)
    print(f"\n{resp}")